# Directed Acyclic Graphs

> **Reference:** *Causal Inference: The Mixtape*, Chapter 3: Directed Acyclic Graphs (pp. 67-117)

This lecture introduces directed acyclic graphs (DAGs) as a tool for reasoning about causal relationships. We apply these concepts using the Online Retail Simulator to answer: **Why does our naive analysis suggest content optimization hurts sales?**

---

## Part I: Theory

This section covers the theoretical foundations of directed acyclic graphs as presented in Cunningham's *Causal Inference: The Mixtape*, Chapter 3.

In [ ]:
# Standard library
import inspect

# Third-party packages
from IPython.display import Code

# Local imports
from support import draw_police_force_example, simulate_police_force_data

## 1. Introduction to DAG notation

A **directed acyclic graph (DAG)** is a visual representation of causal relationships between variables. Each **node** stands for a random variable, and each **arrow** stands for a direct causal effect of one variable on another.

### Key properties

1. **Directed**: Arrows point in one direction (cause → effect)
2. **Acyclic**: No variable can cause itself (no loops)
3. **Causality flows forward**: Time moves in the direction of arrows

### What DAGs encode

DAGs encode **qualitative causal knowledge**. The drawn arrows state which variables cause which, and the missing arrows matter just as much. A missing arrow from $A$ to $B$ claims that $A$ does not directly cause $B$.

### Simple DAG: treatment — outcome

![Simple Causal Relationship](dag_simple.svg)

### Naming the parts of a graph

The rest of the lecture describes graphs with a small vocabulary for the parts of a graph, for the nodes in it, and for the paths between nodes. Within each table, every meaning completes the phrase in the column header. Later sections refer back to these tables.

**The graph**

| Term | A graph element that … | Example |
|------|------------------------|---------|
| Node | stands for a random variable | Education |
| Arrow | claims that one variable directly causes another | Ability → education |
| Missing arrow | claims that one variable does not directly cause another | No arrow from education to ability |

**Nodes**

| Term | A node that … | Example |
|------|---------------|---------|
| Treatment $D$ | causes the effect we want to learn | Education |
| Outcome $Y$ | shows the effect we measure | Earnings |
| Parent of $B$ | has an arrow pointing into $B$ | Ability is a parent of education |
| Child of $A$ | receives an arrow from $A$ | Education is a child of ability |
| Ancestor of $B$ | can reach $B$ by following arrows forward | Ability is an ancestor of earnings |
| Descendant of $A$ | can be reached from $A$ by following arrows forward | In $D \to A \leftarrow B$, $A$ is a descendant of $D$ but $B$ is not |
| Observed | is measured in the data, drawn with a solid border | Family background |
| Unobserved | is not measured, drawn with a dashed border | Ability |
| Conditioned on | we hold fixed by comparing units with the same value, drawn inside a box | Tickets in the M-bias example |
| Confounder | causes both $D$ and $Y$, directly or through other nodes, without passing through $D$ | Ability |
| Mediator | lies on a causal path from $D$ to $Y$ | Occupation |

**Paths**

| Term | A path that … | Example |
|------|---------------|---------|
| Path | connects two nodes through a sequence of arrows, whatever their direction | $D \leftarrow X \to Y$ |
| Causal path | runs from $D$ to $Y$ with every arrow pointing toward $Y$ | $D \to Y$ |
| Backdoor path | connects $D$ and $Y$ and starts with an arrow into $D$ | $D \leftarrow X \to Y$ |
| Chain at $M$ | passes through $M$ with one arrow in and one arrow out | $D \to M \to Y$ |
| Fork at $X$ | leaves $X$ along both of its arrows at $X$ | $D \leftarrow X \to Y$ |
| Collider at $C$ | enters $C$ along both of its arrows at $C$ | $D \to C \leftarrow Y$ |
| Open path | transmits association between its two end points | $D \leftarrow X \to Y$ with $X$ left alone |
| Blocked path | does not transmit association between its two end points | $D \leftarrow X \to Y$ with $X$ conditioned on |

The same three shapes, chain, fork, and collider, can occur at any node along a longer path, not only between $D$ and $Y$. The M-bias example in Section 4 shows a collider far from both.

## 2. Why backdoor paths bias comparisons

Section 1 distinguished causal paths, which run from $D$ to $Y$ along the arrows, from backdoor paths, which start with an arrow into $D$.

### The backdoor problem

Backdoor paths can create **spurious correlations** between $D$ and $Y$:
- They make $D$ and $Y$ appear related even without a causal effect
- This is the graphical representation of **selection bias**, the baseline bias term from the potential outcomes lecture

![Confounded Relationship](dag_confounder.svg)

### Path analysis

| Path | Type |
|------|------|
| $D \to Y$ | Direct causal path (what we want to estimate) |
| $D \leftarrow X \to Y$ | Backdoor path (creates bias) |

### Open and blocked paths

Association travels along paths. An **open** path transmits association between its two end points, and a **blocked** path, also called a closed path, does not. The naive comparison $E[Y \mid D=1] - E[Y \mid D=0]$ picks up the association from every open path between $D$ and $Y$. The causal path carries the effect we want to measure, and every open backdoor path adds spurious association on top of it. The goal is therefore to keep the causal paths open and to block every backdoor path.

### Conditioning: the tool we control

We cannot change the graph, but we can choose what to condition on. To **condition** on a variable $Z$ means to compare $D$ and $Y$ only among units that share the same value of $Z$. The comparison moves from $E[Y \mid D=1] - E[Y \mid D=0]$ to $E[Y \mid D=1, Z=z] - E[Y \mid D=0, Z=z]$ for each value $z$. Conditioning takes several familiar forms:

- **Stratifying**: comparing $D$ and $Y$ within groups defined by $Z$
- **Matching**: pairing treated and control units with similar values of $Z$
- **Restricting the sample**: keeping only units with one value of $Z$
- **Adding a regression control**: including $Z$ as a covariate

Conditioning changes which paths are open. On a chain or a fork, it closes the path, because within a group that shares a value of $Z$ the variable cannot vary and so cannot pass association along. On a collider it does the opposite and opens the path. Only observed variables can be conditioned on, which is why DAGs mark unobserved variables with a dashed border. The figures in this lecture draw a box around a node that we condition on.

The rest of Part I follows what conditioning does. Section 3 shows conditioning on a confounder closing a backdoor path, Section 4 shows conditioning on a collider opening one, Section 5 turns these rules into the **backdoor criterion**, a check on the graph that tells us which variables to condition on, and Section 6 shows conditioning that backfires.

The content optimization puzzle from the introduction is a backdoor path problem of exactly this kind. Part II returns to it.

## 3. Confounders

A **confounder** is a variable that:
1. Causes the treatment ($D$)
2. Causes the outcome ($Y$)
3. Is not on the causal path from $D$ to $Y$

Both causes can be indirect. A variable that reaches $D$ through a chain of arrows, such as $B \to A \to D$, still causes $D$. What matters for the outcome is that the variable reaches $Y$ along a directed path that does not pass through $D$.

### Example: education and earnings

Consider estimating the return to education, the effect of schooling on later earnings. People with higher ability tend to get more education and to earn more regardless of education, and family background works the same way.

- **Treatment** ($D$): Years of education
- **Outcome** ($Y$): Earnings
- **Confounder** ($A$): Ability, unobserved
- **Confounder** ($F$): Family background, observed
- **Form of conditioning**: Stratifying by family background or adding it as a regression control

![Education and Earnings DAG](dag_education.svg)

**Should we condition on family background, and is that enough?**

The graph has two backdoor paths, $D \leftarrow F \to Y$ and $D \leftarrow A \to Y$, and both are open when nothing is conditioned on. Conditioning on $F$ blocks the first. The second stays open, because $A$ is unobserved and cannot be conditioned on, so comparing people with the same background still overstates the return to education.

A confounder we cannot observe is a problem that conditioning alone cannot solve.

## 4. Colliders and collider bias

A **collider** is a node where two arrows on a path point into it:

![Collider Structure](dag_collider.svg)

Colliders have a special property. A collider blocks the path it sits on without any conditioning, and conditioning on it opens that path.

### Why conditioning opens colliders

Conditioning on a collider makes its causes appear correlated, even if they're independent in the population.

Suppose a marketplace features a product on its home page if the product has either great photos or great reviews, and that photo quality and review quality are unrelated across all products. Among featured products, a product with weak photos must have great reviews, or it would not have been featured. Within the featured group, photos and reviews therefore look negatively related, although no such relationship exists in the full catalog.

### Example: M-bias in content optimization

A collider can also sit on a backdoor path, and then it is the reason that path does no harm. Consider a company that optimizes content mainly for products whose sellers are highly engaged, because engaged sellers ask for it. Engaged sellers also file more support tickets. Products with fulfillment problems generate tickets as well, and fulfillment problems lower sales. The number of tickets itself has no effect on sales.

- **Treatment** ($D$): Content optimization
- **Outcome** ($Y$): Sales
- **Cause of the treatment** ($A$): Seller engagement
- **Cause of the outcome** ($B$): Fulfillment problems
- **Collider** ($C$): Support tickets, recorded before the optimization
- **Form of conditioning**: Adding support tickets as a regression control

![M-Bias DAG](dag_mbias.svg)

**Should we condition on support tickets?**

The backdoor path $D \leftarrow A \to C \leftarrow B \to Y$ passes through tickets, and both arrows on it point into $C$. With nothing conditioned on, the collider blocks the path, and because the graph has no other backdoor path, the naive comparison is unbiased. An analyst who controls for tickets, because they look like a reasonable measure of product health, opens the path. In the graph, that analyst boxes $C$, and no node on the path blocks it anymore.

Among products with the same number of tickets, those with highly engaged sellers must have fewer fulfillment problems. Engagement drives optimization, so optimized products now differ from the rest in their fulfillment problems, and the comparison is biased. Adding $A$ or $B$ to the conditioning set closes the path again, because neither is a collider on it.

This structure is called **M-bias** after the shape of the graph. Being measured before treatment does not make a variable a safe control. What matters is where the variable sits on the paths between $D$ and $Y$.

## 5. The backdoor criterion

### Why the criterion matters

Section 2 showed that open backdoor paths add spurious association to the naive comparison. In the language of the potential outcomes lecture, they are the source of the baseline bias term $E[Y^0 \mid D=1] - E[Y^0 \mid D=0]$. If a set of variables $Z$ blocks every backdoor path, then treatment is as good as randomly assigned within groups that share a value of $Z$:

$$(Y^1, Y^0) \perp\!\!\!\perp D \mid Z$$

The comparison within those groups then identifies the causal effect. The matching lecture calls this the conditional independence assumption and builds estimators on it.

The backdoor criterion turns the question of what to condition on into a check we can carry out on the graph.

### Definition

A set of variables $Z$ satisfies the backdoor criterion relative to $(D, Y)$ if:

1. No variable in $Z$ is a descendant of $D$
2. $Z$ blocks every backdoor path from $D$ to $Y$

The first condition excludes variables that the treatment itself affects, for two reasons. Conditioning on a mediator blocks part of the effect we want to measure. Conditioning on any descendant of $D$ can also open a collider path that starts at $D$, such as $D \to C \leftarrow Y$. Section 6 shows both problems in practice.

To check the second condition, we need to know when a set of variables blocks a path.

### How to block paths

A path passes through each of its nodes as a chain, a fork, or a collider (Section 1). Each shape responds differently to conditioning:

| Role | Structure | Intuition | Left alone | Conditioned on |
|------|-----------|-----------|------------|----------------|
| **Chain** (mediator) | $D \to M \to Y$ | Changes in $D$ pass through $M$ to $Y$ | Open | Blocked |
| **Fork** (common cause) | $D \leftarrow X \to Y$ | $X$ moves $D$ and $Y$ together | Open | Blocked |
| **Collider** | $D \to C \leftarrow Y$ | $D$ and $Y$ are separate causes of $C$ | Blocked | Open |

Conditioning on a descendant of a collider also opens the path, though usually less strongly than conditioning on the collider itself.

### Reading long paths

A node blocks a path in one of two ways. Either it is a chain or fork node that we condition on, or it is a collider that we leave alone, with none of its descendants conditioned on. A path is blocked as soon as one node on it blocks it. Three consequences make long paths easy to read:

1. **Roles belong to paths, not to variables**: The same variable can be a chain on one path and a collider on another. Section 6 gives an example.
2. **Colliders close long paths**: A path that passes through a collider is closed unless we condition on that collider or one of its descendants. Long zig-zag paths usually contain one.
3. **Chains and forks stay open**: A backdoor path made only of chains and forks, such as $D \leftarrow A \leftarrow B \to Y$, stays open until we condition on one of its nodes. Conditioning on either $A$ or $B$ closes it.

Conditioning on more variables is therefore not automatically safer, because conditioning on a collider or on a descendant of $D$ can create bias. We only need enough variables to block every backdoor path, and often several different sets do the job.

### Applying the criterion

The M-bias example in Section 4 has one backdoor path, $D \leftarrow A \to C \leftarrow B \to Y$, which the collider $C$ blocks. The empty set already satisfies the criterion, and so do $\{A\}$ and $\{B\}$, so the naive comparison is unbiased. The set $\{C\}$ fails because it opens the path, while $\{A, C\}$ and $\{B, C\}$ close it again. In the education example in Section 3, the set $\{F\}$ fails because the backdoor path $D \leftarrow A \to Y$ stays open. Ability is unobserved, so no set of observed variables satisfies the criterion.

The criterion is sufficient for identification but not necessary. When no observed set satisfies it, as in the education example, we need a different strategy, such as randomization, an instrumental variable, or the front-door criterion. Part II applies the criterion step by step, and the practice cards linked at the end of Part I offer more graphs.

## 6. Collider bias in practice

Both examples in this section condition on a variable that is at once a descendant of the treatment and a collider. They show why the first condition of the backdoor criterion excludes descendants of $D$.

The two examples reach that variable in different ways. In the discrimination example, an analyst controls for occupation on purpose, and occupation is both a mediator and a collider. In the police example, nobody chooses to condition on anything. The way the data are collected does it.

### Example: discrimination and occupation

Consider studying gender discrimination in wages. Women may be steered into lower-paying occupations, and they may also be paid less than men in the same occupation. Ability affects both the occupation a person reaches and what they earn, and it is unobserved.

- **Treatment** ($D$): Gender
- **Outcome** ($Y$): Wages
- **Mediator and collider** ($M$): Occupation
- **Common cause** ($A$): Ability, unobserved
- **Form of conditioning**: Comparing men and women within the same occupation (stratifying)

![Gender Discrimination DAG](dag_discrimination.svg)

**Should we control for occupation?**

The answer depends on which effect we want. $D$ has no arrows pointing into it, so there are no backdoor paths, and the unconditional comparison identifies the **total effect** of discrimination through both channels, $D \to Y$ and $D \to M \to Y$.

The **direct effect** asks how much women earn less than men in the same occupation. Conditioning on $M$ seems to answer it, because it blocks the mediated path $D \to M \to Y$. But $M$ is also a collider on $D \to M \leftarrow A \to Y$, and conditioning on it opens that path. Women who reach a high-paying occupation despite being steered away from it tend to have higher ability than men in the same occupation. Their higher ability raises their wages and masks part of the discrimination they face.

Without a measure of ability, no conditioning set identifies the direct effect.

### Example: police use of force

Consider studying whether police use more force against minorities. Minorities are more likely to be stopped, suspicion affects both whether someone is stopped and how much force is used, and administrative data only record people who were stopped.

- **Treatment** ($D$): Minority status
- **Outcome** ($Y$): Use of force
- **Collider** ($S$): Police stop, which decides who appears in the data
- **Common cause** ($U$): Suspicion or perceived threat, unobserved
- **Form of conditioning**: Restricting the sample to stopped individuals

![Police Use of Force DAG](police_force_dag.svg)

**Do police records reveal discrimination?**

Stop is a descendant of minority status and a collider on $D \to S \leftarrow U \to Y$. This path leaves $D$ through an outgoing arrow, so it is not a backdoor path, and the collider keeps it closed. Records that only cover stopped individuals condition on $S$ and open it.

![Police Use of Force DAG, conditioning on the stop](police_force_dag_conditioned.svg)

Among stopped individuals $(S = 1)$, non-minorities $(D = 0)$ probably had high suspicion $(U)$, since that is why they were stopped. Minorities $(D = 1)$ who were stopped could have low or high suspicion, because they face higher stop rates regardless. Within the stopped sample, non-minorities are therefore disproportionately high-suspicion, which goes with more force $(Y)$. This narrows the apparent gap between the groups and masks the true discrimination effect $(D \to Y)$.

Restricting the sample is conditioning, even though no variable is ever added to a model.

#### Simulation setup

Because we build the data ourselves, we know the true discrimination effect and can see how restricting the sample to stopped individuals distorts it. The following function generates synthetic data with the collider structure described above:

In [ ]:
Code(inspect.getsource(simulate_police_force_data), language="python")

In [ ]:
# Simulate population data
police_data = simulate_police_force_data(n_population=5000, discrimination_effect=0.3, seed=42)
police_data

In [ ]:
# Visualize collider bias
draw_police_force_example(police_data)

#### Collider bias in action

**True discrimination effect:** minority status adds 6 points to the force score, about 24% of the average for non-minorities (25 points)

| Sample | Correlation (Minority ↔ Force) |
|--------|-------------------------------|
| Full population | $r = 0.22$ |
| Stopped individuals only | $r = 0.12$ |

Restricting the data to stopped individuals cuts the correlation almost in half, so an analysis of police records alone underestimates discrimination. In this simulation the stop itself has no effect on force, so the drop is **collider bias** alone. Restricting the sample removes no part of the causal effect. It only opens the path through suspicion.

To practice reading paths and applying the backdoor criterion, work through the [DAG practice cards](practice.ipynb).

---

## Part II: Application

In Part I we developed the theory of directed acyclic graphs. Confounders open backdoor paths, conditioning on a variable closes the paths that run through it unless it is a collider, and the **backdoor criterion** tells us which variables to condition on. **Collider bias** shows how conditioning on the wrong variable can create spurious associations.

We now apply DAG concepts to diagnose and solve a confounding problem using simulated data from the Online Retail Simulator. The simulator provides both potential outcomes—an omniscient view that we would not have in real data—enabling numerical verification of the theoretical results from Part I.

### From unconditional to conditional comparisons

A **naive estimator** compares average outcomes between treated and untreated groups:

$$\hat{\delta}_{\text{naive}} = E[Y \mid D=1] - E[Y \mid D=0]$$

This fails when confounders create systematic differences between groups. The treated and untreated have different distributions of background characteristics—we're comparing apples to oranges.

The **backdoor criterion** tells us what to condition on. Once we block all backdoor paths, a conditional comparison becomes valid:

$$\hat{\delta}_{\text{conditional}} = E[Y \mid D=1, X] - E[Y \mid D=0, X]$$

The key insight: the problem isn't the method (comparing means), it's *what* we compare. Conditioning on the right variables lets us compare like with like.

In [ ]:
# Third-party packages
import pandas as pd
from online_retail_simulator import simulate, load_job_results

# Local imports
from support import (
    compute_effects,
    create_confounded_treatment,
    plot_conditional_comparison,
    plot_confounding_bar,
)

## 1. Business context: the content optimization paradox

An e-commerce company ran a content optimization program for some of its products. When they analyze the results, they find something puzzling:

> **Products that received content optimization tend to have LOWER sales than those that didn't.**

The content team is confused. Did their optimization work actually hurt sales?

### The underlying reality

What's actually happening:
- **Struggling products** (low quality) were selected for content optimization
- **Strong products** (high quality) sell well without optimization
- Content optimization **does** increase sales (true causal effect is positive)

But the **confounding** from product quality creates a **negative spurious correlation** that overwhelms the positive causal effect. We'll use DAGs to understand and solve this problem.

| Variable | Notation | Description |
|----------|----------|-------------|
| Treatment | $D$ | Content optimization (1 = optimized, 0 = not) |
| Outcome | $Y$ | Product revenue |
| Confounder | $X$ | Product quality (High / Low) |
| Potential outcomes | $Y^0, Y^1$ | Revenue without / with optimization |

## 2. Drawing the DAG

Let's represent this situation graphically:

- **Quality** ($Q$): Product quality/strength (unobserved)
- **Optimization** ($D$): Content optimization treatment
- **Sales** ($Y$): Revenue

Relationships:
1. Quality → Sales (+): Better products sell more
2. Quality → Optimization (−): Struggling products get optimized first
3. Optimization → Sales (+): Optimization increases sales (TRUE causal effect)

![Content Optimization Paradox DAG](dag_optimization.svg)

### Path analysis

| Path | Type | Effect |
|------|------|--------|
| Optimization → Sales | Direct (causal) | True causal effect (+50% revenue boost) |
| Optimization ← Quality → Sales | Backdoor | Creates negative bias (quality confounding) |

## 3. Generating data with the Online Retail Simulator

We use the **Online Retail Simulator** to generate realistic e-commerce data. The simulation configuration is defined in `"config_simulation.yaml"`. This gives us products with baseline sales metrics that we can then use to demonstrate confounding.

### Data generation process

1. **Simulate baseline data**: Generate products and their sales metrics
2. **Create quality score**: Derive a quality measure from baseline revenue (the confounder)
3. **Apply confounded treatment**: Assign content optimization based on quality (not randomly!)
4. **Calculate outcomes**: Apply the true treatment effect to get observed sales

In [ ]:
# Step 1: Generate baseline data using the simulator
! cat "config_simulation.yaml"

In [ ]:
# Run simulation
job_info = simulate("config_simulation.yaml")

In [ ]:
# Load simulation results
metrics = load_job_results(job_info)["metrics"]

print(f"Metrics records: {len(metrics)}")

### The assignment mechanism

Now we create the confounding structure:
1. **Quality**: Binary (High/Low) based on median baseline revenue
2. **Treatment assignment**: Low quality products are more likely to be selected for optimization

This mimics a realistic business scenario where struggling products get prioritized for improvement.

In [ ]:
Code(inspect.getsource(create_confounded_treatment), language="python")

In [ ]:
# Create confounded treatment assignment
# Low quality products more likely to be optimized
TRUE_EFFECT = 0.5  # 50% revenue boost from optimization

confounded_products = create_confounded_treatment(
    metrics,
    prob_treat_low=0.6,  # 60% of low quality products get optimized
    prob_treat_high=0.2,  # 20% of high quality products get optimized
    true_effect=TRUE_EFFECT,
    seed=42,
)

print(f"Total products: {len(confounded_products)}")
print(f"\nQuality distribution:")
print(confounded_products["quality"].value_counts())
print(f"\nTreatment rates by quality:")
print(confounded_products.groupby("quality")["D"].mean().round(2))
print(f"\n-> Low quality products are MORE likely to be treated (confounding!)")

In [ ]:
# Visualize the confounding structure
plot_confounding_bar(confounded_products, TRUE_EFFECT, title="Confounding in Content Optimization")

## 4. What does naive analysis tell us?

Let's start with what a naive analyst might do: compare average sales between optimized and non-optimized products.

In [ ]:
# Naive comparison: unconditional difference in means
treated = confounded_products[confounded_products["D"] == 1]
control = confounded_products[confounded_products["D"] == 0]

naive_estimate = treated["Y_observed"].mean() - control["Y_observed"].mean()

print("Naive Comparison: E[Y|D=1] - E[Y|D=0]")
print("=" * 50)
print(f"Mean revenue (treated):   ${treated['Y_observed'].mean():,.2f}")
print(f"Mean revenue (control):   ${control['Y_observed'].mean():,.2f}")
print(f"Naive estimate:           ${naive_estimate:,.2f}")
print(f"\nTrue effect: +{TRUE_EFFECT:.0%} revenue boost")
print(f"\n-> The naive estimate suggests optimization HURTS sales!")

## 5. How do we apply the backdoor criterion?

### Step 1: List all paths from Optimization to Sales

1. **Optimization → Sales** (direct, causal)
2. **Optimization ← Quality → Sales** (backdoor, non-causal)

### Step 2: Identify which paths are open/closed

- Path 1: Always open (it's causal)
- Path 2: Open because Quality is a non-collider on this path

### Step 3: Find conditioning set to block backdoors

To block the backdoor path **Optimization ← Quality → Sales**:
- Condition on **Quality**

This satisfies the backdoor criterion:
- Quality is not a descendant of Optimization
- Conditioning on Quality blocks the backdoor path

## 6. How do we recover the causal effect?

We condition on quality by computing treatment effects **within each quality bin**, the stratifying form of conditioning from Part I. This is the conditional comparison in action—we're comparing products with the same quality level, some optimized and some not.

**Low Quality:**

$$E[Y \mid D=1, Q=\text{Low}] - E[Y \mid D=0, Q=\text{Low}]$$

**High Quality:**

$$E[Y \mid D=1, Q=\text{High}] - E[Y \mid D=0, Q=\text{High}]$$

The weighted average of these within-bin effects gives us an unbiased estimate of the treatment effect.

In [ ]:
# Compute within-bin treatment effects
effects = compute_effects(confounded_products)

print("Within-Bin Treatment Effects")
print("=" * 50)

for quality in ["Low", "High"]:
    bin_data = confounded_products[confounded_products["quality"] == quality]
    treated_mean = bin_data[bin_data["D"] == 1]["Y_observed"].mean()
    control_mean = bin_data[bin_data["D"] == 0]["Y_observed"].mean()
    n_treated = (bin_data["D"] == 1).sum()
    n_control = (bin_data["D"] == 0).sum()

    print(f"\n{quality} Quality (n={len(bin_data)}):")
    print(f"  E[Y|D=1, Q={quality}] = ${treated_mean:,.2f}  (n={n_treated})")
    print(f"  E[Y|D=0, Q={quality}] = ${control_mean:,.2f}  (n={n_control})")
    print(f"  Effect: ${effects['by_quality'][quality]:,.2f}")

print(f"\n" + "=" * 50)
print(f"Weighted average (conditional estimate): ${effects['conditional']:,.2f}")
print(f"Naive estimate:                          ${effects['naive']:,.2f}")
print(f"True effect: +{TRUE_EFFECT:.0%} revenue boost")
print(f"\n-> Within-bin comparisons recover the POSITIVE effect!")

In [ ]:
# Visual summary
plot_conditional_comparison(confounded_products, TRUE_EFFECT)

In [ ]:
true_effect_dollars = confounded_products["baseline_revenue"].mean() * TRUE_EFFECT

summary = pd.DataFrame(
    {
        "Method": ["Naive (unconditional)", "Conditional (within-bin avg)"],
        "Estimate ($)": [effects["naive"], effects["conditional"]],
        "Error ($)": [
            effects["naive"] - true_effect_dollars,
            effects["conditional"] - true_effect_dollars,
        ],
        "% Error": [
            (effects["naive"] - true_effect_dollars) / true_effect_dollars * 100,
            (effects["conditional"] - true_effect_dollars) / true_effect_dollars * 100,
        ],
    }
)
summary["Estimate ($)"] = summary["Estimate ($)"].map(lambda x: f"${x:,.2f}")
summary["Error ($)"] = summary["Error ($)"].map(lambda x: f"${x:,.2f}")
summary["% Error"] = summary["% Error"].map(lambda x: f"{x:+.1f}%")

print(f"True causal effect: +{TRUE_EFFECT:.0%} revenue boost (${true_effect_dollars:,.2f})")
print()
summary

## Additional resources

- **Bellemare, M. & Bloem, J. (2020)**. The paper of how: Estimating treatment effects using the front-door criterion. *Working Paper*.

- **Hünermund, P. & Bareinboim, E. (2019)**. Causal inference and data-fusion in econometrics. *arXiv preprint arXiv:1912.09104*.

- **Imbens, G. W. (2020)**. Potential outcome and directed acyclic graph approaches to causality: Relevance for empirical practice in economics. *Journal of Economic Literature*, 58(4), 1129-1179.

- **Manski, C. F. (1995)**. *Identification Problems in the Social Sciences*. Harvard University Press.

- **Morgan, S. L. & Winship, C. (2014)**. *Counterfactuals and Causal Inference*. Cambridge University Press.

- **Pearl, J. (2009a)**. *Causality: Models, Reasoning, and Inference* (2nd ed.). Cambridge University Press.

- **Pearl, J. (2009b)**. Causal inference in statistics: An overview. *Statistics Surveys*, 3, 96-146.

- **Pearl, J. (2012)**. The do-calculus revisited. *Proceedings of the 28th Conference on Uncertainty in Artificial Intelligence*.

- **Peters, J., Janzing, D. & Schölkopf, B. (2017)**. *Elements of Causal Inference: Foundations and Learning Algorithms*. MIT Press.